# BetterCallNLI — MS3 Evaluation on Kaggle (local base model)

Runs the full multi-agent pipeline against a **base** Qwen2.5 model loaded locally on a Kaggle GPU. No HuggingFace Serverless API calls — every LLM invocation hits the GPU directly via `LocalInferenceClient`.

## Spec compliance (MS3 §2f)

> *Do not use the fine-tuned model in your implementation, you can use any one or more of the models from the same family of the fine-tuned model.*

This notebook loads **`Qwen/Qwen2.5-7B-Instruct`** (the base instruction-tuned model — same family as the MS1 fine-tune, no LoRA adapter attached). DO NOT change `MODEL_NAME` to point at your `qwen3-4B-nli-lora-adapter` — that violates §2f.

## Required Kaggle setup

1. **Accelerator:** GPU T4 x2 (or any single T4 — the 4-bit Qwen2.5-7B fits in ~10 GB)
2. **Add Input → Datasets:** upload your zipped BetterCallNLI repo as a Kaggle Dataset
3. **Add-ons → Secrets:**
   - `CHROMA_API_KEY` (for `RETRIEVAL_MODE = 'vector'`)
   - `NEO4J_URI` / `NEO4J_USERNAME` / `NEO4J_PASSWORD` (for `RETRIEVAL_MODE = 'graphrag'`)
   - `HF_TOKEN` (optional — only needed if Qwen weights download requires gated access)

## What this produces

- `predictions_ms3.json` — verdicts per contract (§3a, §3b)
- `runtraces/runtrace_<id>.json` — one schema-compliant runtrace per contract (§2c, §2h, §3d)
- `evaluation_metrics_ms3.csv` — MS3 metrics (§3e)
- `evaluation_metrics_combined.csv` — MS1 + MS3 side-by-side (§5b)
- `runtraces_ms3.zip` — zipped runtraces (§5c deliverable)

## Cell 1 · Install dependencies

In [ ]:
# Unsloth + bitsandbytes for 4-bit Qwen loading on T4
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q -U 'bitsandbytes>=0.46.1'

# Pipeline deps
!pip install -q rich tqdm pandas pyyaml python-dotenv \
  huggingface_hub sentence-transformers chromadb neo4j kagglehub

## Cell 2 · Configure paths + secrets

In [ ]:
import os, sys
from pathlib import Path

# ---- UPDATE if your dataset mount path differs -----------------------------
REPO_DIR      = Path('/kaggle/input/bettercallnli/BetterCallNLI')
PLAYBOOK_PATH = REPO_DIR / 'playbook.yaml'
MS1_CSV_PATH  = REPO_DIR / 'results' / 'evaluation_metrics.csv'   # for §5b combined CSV

# ---- Model selection (BASE only — no fine-tune per §2f) --------------------
# Options that fit on a T4 (16 GB):
#   - 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'  (pre-quantized, fastest load)
#   - 'Qwen/Qwen2.5-7B-Instruct'              (Unsloth quantizes on load)
# Bigger same-family base options if you have access to A100/L4 sessions:
#   - 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 2048

# ---- Eval options ----------------------------------------------------------
RETRIEVAL_MODE  = 'vector'        # 'vector' or 'graphrag'
LIMIT_CONTRACTS = None             # set int (e.g. 5) for a smoke run
OUTPUT_DIR      = Path('/kaggle/working/outputs/ms3')

# ---- Single GPU ------------------------------------------------------------
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ---- Pull Kaggle Secrets into env ------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ('HF_TOKEN', 'CHROMA_API_KEY',
              'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD'):
        try:
            os.environ[k] = secrets.get_secret(k)
        except Exception:
            pass
    have = [k for k in ('HF_TOKEN', 'CHROMA_API_KEY', 'NEO4J_URI') if os.getenv(k)]
    print('Loaded secrets:', have)
except ImportError:
    print('kaggle_secrets not available; assuming env vars are already set')

# Stub a placeholder HF_TOKEN so build_orchestrator doesn't bail.
# The shim ignores it; it's only checked at orchestrator-construction time.
os.environ.setdefault('HF_TOKEN', 'local-model-no-api-call')

sys.path.insert(0, str(REPO_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_DIR   :', REPO_DIR, '(exists)' if REPO_DIR.exists() else '(MISSING)')
print('PLAYBOOK   :', PLAYBOOK_PATH, '(exists)' if PLAYBOOK_PATH.exists() else '(MISSING)')
print('MS1 CSV    :', MS1_CSV_PATH, '(exists)' if MS1_CSV_PATH.exists() else '(NOT FOUND — combined CSV will only have MS3 row)')
print('MODEL_NAME :', MODEL_NAME)
print('OUTPUT_DIR :', OUTPUT_DIR)

## Cell 3 · Load the base model on GPU

Unsloth loads Qwen2.5-7B-Instruct in 4-bit (~6–8 GB VRAM on a T4). NO LoRA adapter is attached — this is the unmodified base model, satisfying §2f.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,
    device_map      = {'': 0},
)
FastLanguageModel.for_inference(model)

print('Model loaded — no adapter attached (base only, §2f compliant).')
print('  name_or_path:', getattr(model, 'name_or_path', '?'))
print('  device      :', next(model.parameters()).device)
print('  dtype       :', next(model.parameters()).dtype)
print('  vram (gb)   :', round(torch.cuda.memory_allocated() / 1024**3, 2))

## Cell 4 · Install the LocalInferenceClient shim

Replaces `huggingface_hub.InferenceClient` globally and patches each already-imported agent module so every agent's `chat_completion(...)` call routes to the GPU model instead of HF Serverless.

In [ ]:
from src.agent.local_inference_client import install_as_global_client

install_as_global_client(model, tokenizer, device='cuda')

# Sanity check: a fresh InferenceClient() now returns the local shim
from huggingface_hub import InferenceClient
client = InferenceClient(model='ignored-by-shim', token='ignored-by-shim')
out = client.chat_completion(
    messages=[
        {'role': 'system', 'content': 'You are a JSON-only assistant.'},
        {'role': 'user',   'content': 'Reply with: {"ok": true}'},
    ],
    max_tokens=32,
    temperature=0.0,
)
print('shim class  :', type(client).__name__)
print('output      :', repr(out.choices[0].message.content))
print('usage       :', out.usage)

## Cell 5 · Build orchestrator + smoke test

In [ ]:
from src.agent.orchestrator import build_orchestrator
from src.agent.history import ConversationHistory
from src.utils.contract_loader import get_test_contracts

orchestrator = build_orchestrator(
    retrieval_mode = RETRIEVAL_MODE,
    playbook_path  = str(PLAYBOOK_PATH),
    model          = MODEL_NAME,  # only used for runtrace metadata; LLM is local
)
print('Orchestrator built.')
print('  retrieval :', orchestrator.retriever.mode)
print('  router    :', type(orchestrator.router._client).__name__, '(should be LocalInferenceClient subclass)')

contracts = get_test_contracts()  # kagglehub download
print(f'Loaded {len(contracts)} test contracts.')

# 1-contract smoke run
import time
smoke = contracts[0]
if hasattr(orchestrator, 'reset_session'):
    orchestrator.reset_session()
t0 = time.perf_counter()
result = orchestrator.run(
    contract     = smoke,
    user_message = 'analyze this contract',
    history      = ConversationHistory(),
)
print(f'\nSmoke run on {smoke["id"]} took {time.perf_counter()-t0:.1f}s — {len(result.get("verdicts", []))} verdicts')
for v in result.get('verdicts', [])[:3]:
    print(f'  {v["hypothesis_id"]:>4}  {v["label"]:<14}  conf={v["confidence"]:.2f}  ev={len(v.get("evidence",[]))}')
print('runtrace present:', bool(result.get('runtrace')))

## Cell 6 · Run full evaluation on the test split

In [ ]:
import json
from scripts.evaluate_ms3 import run_evaluation
from tqdm.auto import tqdm

total = LIMIT_CONTRACTS or len(contracts)
bar = tqdm(total=total, desc='evaluate')

def _progress(i, n, c_id, *, status='ok', latency_ms=0.0):
    bar.set_postfix_str(f'{c_id} [{status}] {latency_ms/1000:.1f}s')
    bar.update(1)

metrics = run_evaluation(
    orchestrator  = orchestrator,
    contracts     = contracts,
    output_dir    = OUTPUT_DIR,
    limit         = LIMIT_CONTRACTS,
    progress_cb   = _progress,
    playbook_path = PLAYBOOK_PATH,
    ms1_csv_path  = MS1_CSV_PATH if MS1_CSV_PATH.exists() else None,
)
bar.close()

print('\n=== Aggregate metrics ===')
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_hypothesis'}, indent=2))

## Cell 7 · Per-hypothesis breakdown + output listing

In [ ]:
import pandas as pd

rows = [
    {'hypothesis': h, 'correct': v['correct'], 'total': v['total'], 'accuracy': v['accuracy']}
    for h, v in sorted(metrics['per_hypothesis'].items())
]
df = pd.DataFrame(rows).sort_values('hypothesis')
print(df.to_string(index=False))

print('\nOutput files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size:,} bytes)')

combined = OUTPUT_DIR / 'evaluation_metrics_combined.csv'
if combined.exists():
    print('\nevaluation_metrics_combined.csv:')
    print(combined.read_text())

## Done

Download these from the right-side *Output* tab:

- `outputs/ms3/runtraces_ms3.zip` — §5c deliverable (one runtrace per contract)
- `outputs/ms3/evaluation_metrics_combined.csv` — §5b deliverable (MS1 + MS3 metrics)